# 24 - Polars: introducción

Una primera toma de contacto con Polars para trabajar con datos tabulares de forma expresiva y reproducible.

## 1. Importar Polars y preparar datos

En este cuaderno usamos únicamente Polars para las operaciones tabulares. Los datos se escriben directamente en el código, así que no necesitamos descargas ni dependencias numéricas adicionales.

In [ ]:
import polars as pl

precios = pl.Series("precios_de_ejemplo", [1.5, 1.2, 2.1])
precios

ventas = pl.DataFrame({
    "producto": ["café", "té", "café", "zumo", "té", "zumo", "café"],
    "categoria": ["bebidas", "bebidas", "bebidas", "bebidas", "bebidas", "bebidas", "bebidas"],
    "unidades": [12, 8, 15, 6, 10, 9, 7],
    "precio": [1.50, 1.20, 1.50, 2.10, 1.20, 2.10, 1.50],
    "valoracion": [4.5, 4.0, None, 4.8, 3.5, 4.2, 4.7],
})

ventas

## 2. Inspeccionar un DataFrame

`shape` indica filas y columnas, `schema` muestra los tipos inferidos y `head` permite consultar las primeras filas.

In [ ]:
print("Forma:", ventas.shape)
print("Esquema:", ventas.schema)
ventas.head(3)

## 3. Seleccionar columnas y crear expresiones

Las expresiones se construyen con `pl.col`. `select` devuelve únicamente las columnas o cálculos solicitados; no modifica el DataFrame original.

In [ ]:
ventas.select(
    pl.col("producto"),
    pl.col("unidades"),
    (pl.col("unidades") * pl.col("precio")).alias("importe"),
)

## 4. Filtrar filas

`filter` conserva las filas que cumplen una condición. Se pueden combinar condiciones con `&` y `|`; cada condición debe ir entre paréntesis.

In [ ]:
ventas.filter(
    (pl.col("unidades") >= 10) &
    (pl.col("precio") < 2)
)

## 5. Añadir columnas derivadas

`with_columns` añade o reemplaza columnas mediante expresiones vectorizadas. La columna `importe` se calcula fila a fila sin escribir bucles.

In [ ]:
ventas = ventas.with_columns(
    (pl.col("unidades") * pl.col("precio")).round(2).alias("importe"),
    pl.col("producto").str.to_uppercase().alias("producto_mayusculas"),
)

ventas

## 6. Tipos, valores nulos y valores distintos

Polars representa los valores ausentes como `null`. Podemos contar nulos, rellenarlos y consultar los valores distintos con expresiones.

In [ ]:
ventas.select(
    pl.col("valoracion").is_null().sum().alias("valoraciones_nulas"),
    pl.col("valoracion").mean().round(2).alias("valoracion_media"),
    pl.col("categoria").n_unique().alias("categorias_distintas"),
)

ventas.with_columns(
    pl.col("valoracion").fill_null(0).alias("valoracion_sin_nulos")
).select("producto", "valoracion", "valoracion_sin_nulos")

## 7. Ordenar resultados

`sort` permite ordenar por una o varias columnas. Aquí mostramos primero las ventas con mayor importe y, en caso de empate, el producto en orden alfabético.

In [ ]:
ventas.sort(
    ["importe", "producto"],
    descending=[True, False],
).select("producto", "unidades", "importe")

## 8. Agregación inicial

Con `group_by` y `agg` resumimos las filas por producto. Cada cálculo recibe un alias para que el resultado sea fácil de leer.

In [ ]:
resumen = (
    ventas
    .group_by("producto")
    .agg(
        pl.col("unidades").sum().alias("unidades_totales"),
        pl.col("importe").sum().round(2).alias("ingresos"),
        pl.col("valoracion").mean().round(2).alias("valoracion_media"),
    )
    .sort("ingresos", descending=True)
)

resumen

## Ideas clave

- Un `DataFrame` contiene datos tabulares y un `Series` representa una columna.
- Las expresiones (`pl.col`, operaciones y alias) describen transformaciones de forma clara.
- `select`, `filter`, `with_columns`, `sort` y `group_by` cubren muchas tareas iniciales.
- En los siguientes cuadernos veremos consultas lazy, combinaciones y operaciones más avanzadas.